# NOTA

1. Si hemos descargado los datos para tener todas las columnas en todos los tipos de taxi, debemos pensar en unificar todos los tipos en un dataframe y pensar como rellenar las columnas que se quedan vacias.
2. Soporte de NaN propios explicado más abajo, sujeto a cambios
3. Añadir tiempo de viaje y franja horaria (para ver horas más concurridas)
4. Podríamos ver la manera de unificar las taxes para que no sean tantas columnas (sumarlas tal vez y dejarlas en una columna como total_taxes, con las taxes faltantes a 0)
5. En relación a lo anterior, añadir columna de si va al aeropuerto, con la tasa de aeropuerto
6. Ver utilidad de passenger_count (en mi opinión sobra)
7. Investigar sobre como los RateCodeID afectan al precio final del viaje
8. "store_and_fwd_flag" nos indica si el vehiculo (o su conductor) estaba conectado a internet en el momento del viaje o del pago. No veo mucha utilidad a esto, se puede discutir

# IMPORTS

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import pyarrow.parquet as pq
from warnings import filterwarnings
filterwarnings("ignore")

In [2]:
path = Path.cwd().parent / "data"

# CARGA DE DATOS

In [5]:
file_path = path / "yellow_taxi_unified.parquet"
pf = pq.ParquetFile(file_path)

frames = []

# Iteramos por trozos (batches) para no llenar la RAM
for batch in pf.iter_batches(batch_size=100_000):
    # Convertimos el trozo a Pandas
    df_chunk = batch.to_pandas()

    # Nos quedamos con el 5% aleatorio de este trozo (frac=0.05)
    sample_chunk = df_chunk.sample(frac=0.05, random_state=42)
    
    frames.append(sample_chunk)

# Unimos las muestras
yellow_df = pd.concat(frames)

print(f"Tamaño de la muestra: {yellow_df.shape}")
print(yellow_df.head())

Tamaño de la muestra: (9722897, 35)
      VendorID     pickup_datetime    dropoff_datetime  passenger_count  \
75721        2 2021-01-03 19:20:08 2021-01-03 20:00:23              5.0   
80184        2 2021-01-04 00:28:41 2021-01-04 00:47:50              1.0   
19864        2 2021-01-01 19:09:55 2021-01-01 19:23:37              1.0   
76699        1 2021-01-03 19:08:28 2021-01-03 19:13:22              0.0   
92991        2 2021-01-04 11:52:36 2021-01-04 12:13:43              1.0   

       trip_distance  RatecodeID store_and_fwd_flag  PULocationID  \
75721          19.98         2.0              False           132   
80184          11.16         1.0              False           138   
19864           2.68         1.0              False           237   
76699           0.90         1.0              False           186   
92991           6.13         1.0              False            24   

       DOLocationID payment_type  ...  sales_tax  driver_pay  \
75721           249            1  

# DATA VIEW

In [7]:
print(f"Yellow Taxi DataFrame Columns: {yellow_df.columns.tolist()}")

Yellow Taxi DataFrame Columns: ['VendorID', 'pickup_datetime', 'dropoff_datetime', 'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag', 'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee', 'dispatching_base_num', 'originating_base_num', 'request_datetime', 'on_scene_datetime', 'bcf', 'sales_tax', 'driver_pay', 'shared_request_flag', 'shared_match_flag', 'access_a_ride_flag', 'wav_request_flag', 'wav_match_flag', 'trip_type', 'ehail_fee', 'hvfhs_license_num']


# YELLOW TAXI

In [8]:
yellow_df.head()

,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,sales_tax,driver_pay,shared_request_flag,shared_match_flag,access_a_ride_flag,wav_request_flag,wav_match_flag,trip_type,ehail_fee,hvfhs_license_num
75721,2,2021-01-03 19:20:08,2021-01-03 20:00:23,5.0,19.98,2.0,False,132,249,1,...,NaN,NaN,None,None,None,None,None,NaN,NaN,NaN
80184,2,2021-01-04 00:28:41,2021-01-04 00:47:50,1.0,11.16,1.0,False,138,177,2,...,NaN,NaN,None,None,None,None,None,NaN,NaN,NaN
19864,2,2021-01-01 19:09:55,2021-01-01 19:23:37,1.0,2.68,1.0,False,237,186,1,...,NaN,NaN,None,None,None,None,None,NaN,NaN,NaN
76699,1,2021-01-03 19:08:28,2021-01-03 19:13:22,0.0,0.90,1.0,False,186,234,1,...,NaN,NaN,None,None,None,None,None,NaN,NaN,NaN
92991,2,2021-01-04 11:52:36,2021-01-04 12:13:43,1.0,6.13,1.0,False,24,249,1,...,NaN,NaN,None,None,None,None,None,NaN,NaN,NaN


Veamos las columnas de los taxis amarillos:

In [7]:
yellow_df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='str')

Vamos a ver los tipos y el número de NaN que hay en las columnas:

In [8]:
for col in yellow_df.columns:
    print(f"{col}:\n\t-col_type: {yellow_df[col].dtype}\n\tcol_nan: {yellow_df[col].isna().sum()}\n")

VendorID:
	-col_type: int32
	col_nan: 0

tpep_pickup_datetime:
	-col_type: datetime64[us]
	col_nan: 0

tpep_dropoff_datetime:
	-col_type: datetime64[us]
	col_nan: 0

passenger_count:
	-col_type: float64
	col_nan: 1014740

trip_distance:
	-col_type: float64
	col_nan: 0

RatecodeID:
	-col_type: float64
	col_nan: 1014740

store_and_fwd_flag:
	-col_type: str
	col_nan: 1014740

PULocationID:
	-col_type: int32
	col_nan: 0

DOLocationID:
	-col_type: int32
	col_nan: 0

payment_type:
	-col_type: int64
	col_nan: 0

fare_amount:
	-col_type: float64
	col_nan: 0

extra:
	-col_type: float64
	col_nan: 0

mta_tax:
	-col_type: float64
	col_nan: 0

tip_amount:
	-col_type: float64
	col_nan: 0

tolls_amount:
	-col_type: float64
	col_nan: 0

improvement_surcharge:
	-col_type: float64
	col_nan: 0

total_amount:
	-col_type: float64
	col_nan: 0

congestion_surcharge:
	-col_type: float64
	col_nan: 1014740

Airport_fee:
	-col_type: float64
	col_nan: 1014740

cbd_congestion_fee:
	-col_type: float64
	col_nan: 0



Vemos que el número de NaN son el mismo en las columnas que tienen, vamos a ver algunos:

In [9]:
yellow_df[yellow_df["passenger_count"].isna()]

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
3166704,2,2025-11-01 00:17:00,2025-11-01 00:21:00,NaN,1.05,NaN,NaN,239,238,0,9.56,0.0,0.5,0.0,0.00,1.0,19.53,NaN,NaN,0.00
3166705,2,2025-11-01 00:27:00,2025-11-01 00:36:00,NaN,1.54,NaN,NaN,142,238,0,14.17,0.0,0.5,0.0,0.00,1.0,18.17,NaN,NaN,0.00
3166706,2,2025-11-01 00:26:54,2025-11-01 00:49:40,NaN,4.41,NaN,NaN,36,112,0,-1.98,0.0,0.5,0.0,0.00,1.0,-0.48,NaN,NaN,0.00
3166707,2,2025-11-01 00:11:00,2025-11-01 00:19:00,NaN,0.68,NaN,NaN,87,87,0,20.17,0.0,0.5,0.0,0.00,1.0,24.17,NaN,NaN,0.00
3166708,2,2025-11-01 00:26:00,2025-11-01 00:57:00,NaN,3.90,NaN,NaN,231,162,0,43.97,0.0,0.5,0.0,0.00,1.0,57.11,NaN,NaN,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4181439,2,2025-11-30 23:12:44,2025-11-30 23:43:26,NaN,10.62,NaN,NaN,68,169,0,33.16,0.0,0.5,0.0,6.94,1.0,44.85,NaN,NaN,0.75
4181440,1,2025-11-30 23:10:35,2025-11-30 23:28:24,NaN,6.50,NaN,NaN,48,116,0,22.17,0.0,0.5,0.0,0.00,1.0,26.92,NaN,NaN,0.75
4181441,2,2025-11-30 23:09:43,2025-11-30 23:36:08,NaN,8.10,NaN,NaN,145,152,0,-4.75,0.0,0.5,0.0,0.00,1.0,4.06,NaN,NaN,0.75
4181442,1,2025-11-30 23:29:41,2025-11-30 23:47:09,NaN,5.60,NaN,NaN,116,48,0,21.42,0.0,0.5,0.0,0.00,1.0,26.17,NaN,NaN,0.75


Al ver estos datos faltantes, podemos transformarlos con lo siguiente:
-    Para los recargos del aeropuerto y el "congestion_surcharge", podemos asumir que son 0, ya que no hay datos sobre ellos.
-    Para el número de pasageros, no son necesarios ya que no varían el precio ni el tiempo de viaje.
-    

In [10]:
yellow_df[["congestion_surcharge", "Airport_fee"]].fillna(0, inplace=True)
yellow_df.drop(columns=["passenger_count"], inplace=True)

In [16]:
yellow_df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2025-11-01 00:13:25,2025-11-01 00:13:25,1.68,1.0,N,43,186,1,14.9,0.00,0.5,1.50,0.00,1.0,22.15,2.5,0.00,0.75
1,2,2025-11-01 00:49:07,2025-11-01 01:01:22,2.28,1.0,N,142,237,1,14.2,1.00,0.5,4.99,0.00,1.0,24.94,2.5,0.00,0.75
2,1,2025-11-01 00:07:19,2025-11-01 00:20:41,2.70,1.0,N,163,238,1,15.6,4.25,0.5,4.27,0.00,1.0,25.62,2.5,0.00,0.75
3,2,2025-11-01 00:00:00,2025-11-01 01:01:03,12.87,1.0,N,138,261,1,66.7,6.00,0.5,0.00,6.94,1.0,86.14,2.5,1.75,0.75
4,1,2025-11-01 00:18:50,2025-11-01 00:49:32,8.40,1.0,N,138,37,2,39.4,7.75,0.5,0.00,0.00,1.0,48.65,0.0,1.75,0.00


# GREEN TAXI

In [11]:
green_df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,1.0,0.74,7.2,...,0.5,1.94,0.0,NaN,1.0,11.64,1.0,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,2.0,0.95,7.2,...,0.5,0.00,0.0,NaN,1.0,9.70,2.0,1.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,1.0,2.19,13.5,...,0.5,5.00,0.0,NaN,1.0,21.00,1.0,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,1.0,5.44,24.7,...,0.5,0.50,0.0,NaN,1.0,27.70,1.0,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,1.0,3.20,18.4,...,1.5,1.00,0.0,NaN,1.0,24.65,1.0,1.0,2.75,0.0


Veamos los tipos de datos y la cantidad de NaN:

In [12]:
for col in green_df.columns:
    print(f"{col}:\n\t-col_type: {green_df[col].dtype}\n\tcol_nan: {green_df[col].isna().sum()}\n")

VendorID:
	-col_type: int32
	col_nan: 0

lpep_pickup_datetime:
	-col_type: datetime64[us]
	col_nan: 0

lpep_dropoff_datetime:
	-col_type: datetime64[us]
	col_nan: 0

store_and_fwd_flag:
	-col_type: str
	col_nan: 5569

RatecodeID:
	-col_type: float64
	col_nan: 5569

PULocationID:
	-col_type: int32
	col_nan: 0

DOLocationID:
	-col_type: int32
	col_nan: 0

passenger_count:
	-col_type: float64
	col_nan: 5569

trip_distance:
	-col_type: float64
	col_nan: 0

fare_amount:
	-col_type: float64
	col_nan: 0

extra:
	-col_type: float64
	col_nan: 0

mta_tax:
	-col_type: float64
	col_nan: 0

tip_amount:
	-col_type: float64
	col_nan: 0

tolls_amount:
	-col_type: float64
	col_nan: 0

ehail_fee:
	-col_type: float64
	col_nan: 46912

improvement_surcharge:
	-col_type: float64
	col_nan: 0

total_amount:
	-col_type: float64
	col_nan: 0

payment_type:
	-col_type: float64
	col_nan: 5569

trip_type:
	-col_type: float64
	col_nan: 5570

congestion_surcharge:
	-col_type: float64
	col_nan: 5569

cbd_congestion_fe

Vemos que en algunas columnas pasa como con los taxis amarillos, pero en otras columnas hay más valores nulos. Vamos a verlos:

In [13]:
green_df[green_df['passenger_count'].isna()]

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
41343,2,2025-11-01 00:46:00,2025-11-01 01:39:00,NaN,NaN,41,80,NaN,9.26,46.92,...,0.5,7.00,0.0,NaN,1.0,58.92,NaN,NaN,NaN,0.75
41344,2,2025-11-01 00:13:00,2025-11-01 00:40:00,NaN,NaN,181,226,NaN,7.01,46.78,...,0.5,9.66,0.0,NaN,1.0,57.94,NaN,NaN,NaN,0.00
41345,6,2025-11-01 00:53:17,2025-11-01 01:16:43,NaN,NaN,50,249,NaN,3.18,2.90,...,0.5,0.00,0.0,NaN,0.3,16.00,NaN,NaN,NaN,0.00
41346,2,2025-11-01 00:43:00,2025-11-01 00:53:00,NaN,NaN,166,116,NaN,1.54,14.38,...,0.5,3.18,0.0,NaN,1.0,19.06,NaN,NaN,NaN,0.00
41347,2,2025-11-01 00:48:00,2025-11-01 01:23:00,NaN,NaN,256,162,NaN,6.06,34.85,...,0.5,0.00,0.0,NaN,1.0,39.85,NaN,NaN,NaN,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
46907,2,2025-11-30 19:58:34,2025-11-30 20:14:28,NaN,NaN,59,51,NaN,8.50,33.22,...,0.5,0.00,0.0,NaN,1.0,34.72,NaN,NaN,NaN,0.00
46908,2,2025-11-30 19:34:00,2025-11-30 19:46:00,NaN,NaN,74,151,NaN,1.73,13.86,...,0.5,0.77,0.0,NaN,1.0,16.13,NaN,NaN,NaN,0.00
46909,2,2025-11-30 21:46:46,2025-11-30 22:17:55,NaN,NaN,33,163,NaN,7.52,38.42,...,0.5,1.00,0.0,NaN,1.0,44.42,NaN,NaN,NaN,0.75
46910,2,2025-11-30 21:00:00,2025-11-30 21:15:00,NaN,NaN,16,95,NaN,5.61,24.67,...,0.5,0.00,0.0,NaN,1.0,26.17,NaN,NaN,NaN,0.00


In [14]:
# Select relevant columns and remove rows with missing values
relevant_cols = ['trip_type', 'trip_distance', 'fare_amount', 'total_amount', 'tip_amount', 'passenger_count']
green_analysis = green_df[relevant_cols].dropna()

# Calculate correlation with trip_type
correlation = green_analysis.corr()['trip_type'].sort_values(ascending=False)
print(correlation)

trip_type          1.000000
fare_amount        0.340749
total_amount       0.303766
tip_amount         0.105139
trip_distance      0.068193
passenger_count    0.038032
Name: trip_type, dtype: float64


Podemos ver varias cosas:
-    En cuanto a "store_and_fwd_flag", "RatecodeID", "passenger_count" y "congestion_surcharge" haremos lo mismo que con los taxis amarillos.
-    Para la "ehail_fee", como no hay datos, le daremos 0.
-    Para el tipo de pago pondremos en efectivo: 0.
-    Vemos que el tipo de viaje no afecta demasiado al resto de variables, por lo que podemos optar por quitarla.

In [15]:
green_df[["congestion_surcharge", "ehail_fee", "payment_type"]].fillna(0, inplace=True)
green_df.drop(columns=["passenger_count", "trip_type"], inplace=True)

In [17]:
green_df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,congestion_surcharge,cbd_congestion_fee
0,2,2025-11-01 00:34:48,2025-11-01 00:41:39,N,1.0,74,42,0.74,7.2,1.00,0.5,1.94,0.0,NaN,1.0,11.64,1.0,0.00,0.0
1,2,2025-11-01 00:18:52,2025-11-01 00:24:27,N,1.0,74,42,0.95,7.2,1.00,0.5,0.00,0.0,NaN,1.0,9.70,2.0,0.00,0.0
2,2,2025-11-01 01:03:14,2025-11-01 01:15:24,N,1.0,83,160,2.19,13.5,1.00,0.5,5.00,0.0,NaN,1.0,21.00,1.0,0.00,0.0
3,2,2025-11-01 00:10:57,2025-11-01 00:24:53,N,1.0,166,127,5.44,24.7,1.00,0.5,0.50,0.0,NaN,1.0,27.70,1.0,0.00,0.0
4,1,2025-11-01 00:03:48,2025-11-01 00:19:38,N,1.0,166,262,3.20,18.4,3.75,1.5,1.00,0.0,NaN,1.0,24.65,1.0,2.75,0.0


# HIGH VOLUME